# Thread Execution & Warps

Understand how GPU threads truly execute in groups of 32 called warps, why branch divergence kills performance, and how occupancy determines hardware utilization.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-thread-execution)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Warps: The True Execution Unit

In [ ]:
!pip install numba

from numba import cuda
import numpy as np
import math

# ------------------------------------------------------------------
# Demo 1: Warp formation -- observe which warp each thread belongs to
# ------------------------------------------------------------------
@cuda.jit
def warp_info_kernel(warp_ids, lane_ids):
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid < warp_ids.size:
        warp_ids[tid] = tid // 32   # warp index
        lane_ids[tid] = tid % 32    # lane within warp

n_threads = 128
warp_ids = cuda.device_array(n_threads, dtype=np.int32)
lane_ids = cuda.device_array(n_threads, dtype=np.int32)

threads_per_block = 64  # 2 warps per block
blocks = math.ceil(n_threads / threads_per_block)

warp_info_kernel[blocks, threads_per_block](warp_ids, lane_ids)

w = warp_ids.copy_to_host()
l = lane_ids.copy_to_host()

print("Thread -> Warp mapping (first 2 blocks of 64 threads):")
for i in range(0, n_threads, 32):
    print(f"  Threads {i:3d}-{i+31:3d}  =>  Warp {w[i]}  (lanes {l[i]}-{l[i+31]})")

# ------------------------------------------------------------------
# Demo 2: Block sizes that are NOT multiples of 32 waste resources
# ------------------------------------------------------------------
print("\n--- Warp utilization for various block sizes ---")
for bs in [32, 64, 96, 100, 128, 160, 200, 256]:
    num_warps = math.ceil(bs / 32)
    active = bs
    total_slots = num_warps * 32
    utilization = active / total_slots * 100
    wasted = total_slots - active
    print(f"  Block size {bs:>4d}: {num_warps} warps, "
          f"{active}/{total_slots} active ({utilization:.1f}%), "
          f"{wasted} wasted slots")

# ------------------------------------------------------------------
# Demo 3: Simple vector add showing warp-level execution
# ------------------------------------------------------------------
@cuda.jit
def vector_add(a, b, c, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        c[idx] = a[idx] + b[idx]

n = 10000
a = np.random.randn(n).astype(np.float32)
b = np.random.randn(n).astype(np.float32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.device_array(n, dtype=np.float32)

TPB = 256  # Always a multiple of 32!
BPG = math.ceil(n / TPB)
vector_add[BPG, TPB](d_a, d_b, d_c, n)

c = d_c.copy_to_host()
print(f"\nVector add: max error = {np.max(np.abs(c - (a + b))):.2e}")
print(f"Launched {BPG} blocks x {TPB} threads = {BPG * TPB} total threads")
print(f"That's {BPG * TPB // 32} warps processing {n} elements")
print(f"(Last block has {BPG * TPB - n} idle threads in boundary warps)")

### Lesson example: Warp Divergence

In [ ]:
!pip install numba

from numba import cuda
import numpy as np
import math
import time

# ------------------------------------------------------------------
# Experiment: Measure the cost of warp divergence
# ------------------------------------------------------------------

# Kernel with data-dependent branching (divergent)
@cuda.jit
def kernel_divergent(data, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        val = data[idx]
        if val > 0.5:
            # Path A: ~50% of random uniform data takes this
            out[idx] = val * val + val * 2.0 + 1.0
        else:
            # Path B: ~50% of random uniform data takes this
            out[idx] = val * 0.5 - val * val + 3.0

# Equivalent branch-free kernel (no divergence)
@cuda.jit
def kernel_branchfree(data, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        val = data[idx]
        # Always execute the same arithmetic (uniform path)
        out[idx] = val * val + val * 2.0 + 1.0

def benchmark_kernel(kernel, d_data, d_out, n, blocks, threads, warmup=5, iters=50):
    for _ in range(warmup):
        kernel[blocks, threads](d_data, d_out, n)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        kernel[blocks, threads](d_data, d_out, n)
    cuda.synchronize()
    elapsed = (time.perf_counter() - start) / iters * 1000  # ms
    return elapsed

# Setup
n = 1_000_000
TPB = 256
BPG = math.ceil(n / TPB)

print("=== Warp Divergence Experiment ===")
print(f"Array size: {n:,}, Blocks: {BPG}, Threads/Block: {TPB}")
print(f"Total warps: {BPG * TPB // 32:,}\n")

# --- Test 1: Random data (maximum divergence within each warp) ---
data_random = np.random.rand(n).astype(np.float32)  # uniform [0, 1)
d_data = cuda.to_device(data_random)
d_out = cuda.device_array(n, dtype=np.float32)

t_div_random = benchmark_kernel(kernel_divergent, d_data, d_out, n, BPG, TPB)
t_bf_random = benchmark_kernel(kernel_branchfree, d_data, d_out, n, BPG, TPB)

print("Test 1: Random data (50/50 branches -> high divergence)")
print(f"  Divergent kernel:   {t_div_random:.4f} ms")
print(f"  Branch-free kernel: {t_bf_random:.4f} ms")
print(f"  Slowdown from divergence: {t_div_random / t_bf_random:.2f}x\n")

# --- Test 2: All-positive data (no divergence, all take same branch) ---
data_positive = np.random.rand(n).astype(np.float32) + 0.6  # all > 0.5
d_data_pos = cuda.to_device(data_positive)

t_div_positive = benchmark_kernel(kernel_divergent, d_data_pos, d_out, n, BPG, TPB)

print("Test 2: All-positive data (all threads take same branch -> no divergence)")
print(f"  Divergent kernel:   {t_div_positive:.4f} ms")
print(f"  Branch-free kernel: {t_bf_random:.4f} ms")
print(f"  Slowdown from divergence: {t_div_positive / t_bf_random:.2f}x")
print(f"  (Should be ~1.0x since no actual divergence occurs)\n")

# --- Test 3: Sorted data (divergence only at boundary warps) ---
data_sorted = np.sort(data_random)  # negatives first, then positives
d_data_sorted = cuda.to_device(data_sorted)

t_div_sorted = benchmark_kernel(kernel_divergent, d_data_sorted, d_out, n, BPG, TPB)

print("Test 3: Sorted data (divergence only at boundary warps)")
print(f"  Divergent kernel:   {t_div_sorted:.4f} ms")
print(f"  Improvement over random: {t_div_random / t_div_sorted:.2f}x")
print(f"  (Sorting reduces divergence to ~1 warp per block at boundary)\n")

# --- Summary ---
print("=" * 55)
print("Summary:")
print(f"  Random data divergence overhead:  {(t_div_random / t_bf_random - 1) * 100:.1f}%")
print(f"  Sorted data divergence overhead:  {(t_div_sorted / t_bf_random - 1) * 100:.1f}%")
print(f"  Uniform data divergence overhead: {(t_div_positive / t_bf_random - 1) * 100:.1f}%")
print("\nLesson: Data distribution determines divergence cost!")

### Lesson example: Occupancy & Resource Limits

In [ ]:
!pip install numba

from numba import cuda
import numpy as np
import math

# ------------------------------------------------------------------
# Demo 1: Query GPU resource limits
# ------------------------------------------------------------------
device = cuda.get_current_device()
print("=== GPU Resource Limits ===")
print(f"Device: {device.name}")
print(f"Max threads per block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max shared memory per block: {device.MAX_SHARED_MEMORY_PER_BLOCK} bytes")
print(f"Warp size: {device.WARP_SIZE}")
print(f"Max block dimensions: {device.MAX_BLOCK_DIM_X} x {device.MAX_BLOCK_DIM_Y} x {device.MAX_BLOCK_DIM_Z}")
print(f"Max grid dimensions: {device.MAX_GRID_DIM_X} x {device.MAX_GRID_DIM_Y} x {device.MAX_GRID_DIM_Z}")
print()

# ------------------------------------------------------------------
# Demo 2: Occupancy calculator (simulated for T4)
# ------------------------------------------------------------------
# T4 GPU specifications (Turing architecture)
t4_config = {
    'name': 'T4 (Turing)',
    'max_threads_per_sm': 1024,
    'regs_per_sm': 65536,
    'shared_mem_per_sm': 65536,  # 64 KB
    'max_blocks_per_sm': 16,
}

def calculate_occupancy(threads_per_block, regs_per_thread,
                        shared_mem_per_block, gpu_config):
    max_warps_per_sm = gpu_config['max_threads_per_sm'] // 32
    warps_per_block = math.ceil(threads_per_block / 32)

    blocks_by_warps = max_warps_per_sm // warps_per_block
    blocks_by_regs = (gpu_config['regs_per_sm'] //
                      (regs_per_thread * threads_per_block)) if regs_per_thread > 0 else 999
    blocks_by_smem = (gpu_config['shared_mem_per_sm'] //
                      shared_mem_per_block) if shared_mem_per_block > 0 else 999
    blocks_by_limit = gpu_config['max_blocks_per_sm']

    active_blocks = min(blocks_by_warps, blocks_by_regs,
                        blocks_by_smem, blocks_by_limit)
    active_warps = active_blocks * warps_per_block
    occupancy = active_warps / max_warps_per_sm

    limiting = "warps"
    if active_blocks == blocks_by_regs:
        limiting = "registers"
    elif active_blocks == blocks_by_smem:
        limiting = "shared memory"
    elif active_blocks == blocks_by_limit:
        limiting = "max blocks"

    return occupancy, active_blocks, active_warps, limiting

print("=== Occupancy Analysis (T4 GPU) ===")
print(f"{'Block Size':<12} {'Regs/Thr':<10} {'SMEM/Blk':<12} {'Blocks/SM':<11} "
      f"{'Warps/SM':<10} {'Occupancy':<12} {'Limiter'}")
print("-" * 85)

scenarios = [
    (256, 32, 0,     "Lightweight kernel"),
    (256, 64, 0,     "Medium register use"),
    (256, 128, 0,    "Heavy register use"),
    (128, 32, 0,     "Small blocks, light regs"),
    (32,  32, 0,     "Tiny blocks (bad!)"),
    (256, 32, 16384, "16 KB shared mem"),
    (256, 32, 32768, "32 KB shared mem"),
    (256, 32, 49152, "48 KB shared mem"),
]

for tpb, rpt, smem, label in scenarios:
    occ, blks, warps, limiter = calculate_occupancy(tpb, rpt, smem, t4_config)
    print(f"{tpb:<12} {rpt:<10} {smem:<12} {blks:<11} {warps:<10} "
          f"{occ*100:>5.1f}%       {limiter} ({label})")

# ------------------------------------------------------------------
# Demo 3: Impact of block size on performance
# ------------------------------------------------------------------
import time

@cuda.jit
def simple_kernel(data, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        val = data[idx]
        out[idx] = val * val + val * 2.0 + 1.0

def bench_block_size(block_size, d_data, d_out, n, warmup=10, iters=100):
    blocks = math.ceil(n / block_size)
    for _ in range(warmup):
        simple_kernel[blocks, block_size](d_data, d_out, n)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        simple_kernel[blocks, block_size](d_data, d_out, n)
    cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1000

n = 2_000_000
d_data = cuda.to_device(np.random.randn(n).astype(np.float32))
d_out = cuda.device_array(n, dtype=np.float32)

print("\n=== Block Size Performance Impact ===")
print(f"{'Block Size':<12} {'Time (ms)':<12} {'Warps/Block':<14} {'Note'}")
print("-" * 55)
for bs in [32, 64, 128, 256, 512, 1024]:
    t = bench_block_size(bs, d_data, d_out, n)
    wpb = bs // 32
    note = "(too small)" if bs < 128 else "(good)" if bs <= 256 else "(large)"
    print(f"{bs:<12} {t:<12.4f} {wpb:<14} {note}")

---

## Exercise: Measuring Warp Divergence Impact


In [ ]:
from numba import cuda
import numpy as np
import math
import time

# --- Kernel 1: Divergent (branches based on data value) ---
@cuda.jit
def kernel_divergent(data, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        val = data[idx]
        # TODO: Add if/else branch based on val > 0.5
        # Branch A: at least 5 FP operations
        # Branch B: at least 5 different FP operations
        pass

# --- Kernel 2: Branch-free (always same path) ---
@cuda.jit
def kernel_branchfree(data, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        val = data[idx]
        # TODO: Same amount of arithmetic, no branches
        pass

def benchmark(kernel, d_data, d_out, n, blocks, threads, warmup=5, iters=20):
    """Time a kernel over multiple iterations, return avg time in ms."""
    # TODO: warmup, synchronize, time, return ms
    pass

# --- Setup ---
n = 2_000_000
TPB = 256
BPG = math.ceil(n / TPB)

# TODO: Create three datasets:
# 1. Random uniform [0, 1)
# 2. All > 0.5 (uniform branch)
# 3. Sorted version of random (boundary-only divergence)

# TODO: Benchmark each kernel with each dataset
# TODO: Print results table showing slowdown from divergence

## Your tasks

Build a controlled experiment to measure the performance cost of warp divergence.

**Requirements:**
1. Implement a 'divergent' kernel that branches based on `data[idx] > 0.5` with different computations in each branch (at least 5 floating-point operations per branch)
2. Implement a 'branch-free' kernel that always executes the same computation path for all threads (same amount of arithmetic)
3. Test with three data distributions:
   - Random uniform [0, 1): ~50% divergence within each warp
   - All values > 0.5: 0% divergence (all threads same branch)
   - Sorted data: divergence only at boundary warps
4. Time each kernel configuration with at least 20 iterations after warmup
5. Report the slowdown factor for each scenario
6. Use array size of at least 1,000,000 elements

**Hints:**
- Use `cuda.synchronize()` before starting and after finishing the timed loop
- The divergent kernel should have meaningfully different computation in each branch (not just +1 vs +2)
- You should observe the slowdown decrease as you go from random -> sorted -> uniform data
- Make sure both kernels produce the same output for uniform input to verify correctness

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-thread-execution) for the solution and discussion.